<a href="https://colab.research.google.com/github/sabharwal-monish/LLM/blob/main/Fine_tuning_using_LoRA_and_SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install deeplake transformers peft accelerate bitsandbytes datasets

In [ ]:
!pip install "deeplake<4"

In [ ]:
import torch
import torch.nn as nn
import deeplake
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# ==========================================
# STEP 1: LOAD & FORMAT DATA
# ==========================================
print(">>> STEP 1: Streaming Data...")
ds = deeplake.load('hub://genai360/GAIR-lima-train-set', read_only=True)
ds_test = deeplake.load('hub://genai360/GAIR-lima-test-set', read_only=True)

In [ ]:
def prepare_sample_text(example):
    return f"Question: {example['question'].text()}\n\nAnswer: {example['answer'].text()}"

# Convert DeepLake to Standard Hugging Face Dataset
# (This avoids the TRL library errors we faced earlier)
def create_standard_dataset(deep_lake_ds):
    data_list = []
    for item in deep_lake_ds:
        data_list.append({"text": prepare_sample_text(item)})
    return Dataset.from_list(data_list)

hf_train_dataset = create_standard_dataset(ds)
hf_eval_dataset = create_standard_dataset(ds_test)
print(f"Data Loaded: {len(hf_train_dataset)} training samples.")


In [ ]:
print(">>> STEP 2: Tokenizing...")
model_id = "facebook/opt-1.3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Fix padding for OPT

def tokenize_function(examples):
    # Truncate to 1024 to fit in memory
    return tokenizer(examples["text"], truncation=True, max_length=1024, padding="max_length")

# Apply tokenization to the standard dataset
tokenized_train = hf_train_dataset.map(tokenize_function, batched=True)
tokenized_eval = hf_eval_dataset.map(tokenize_function, batched=True)

In [ ]:
print(">>> STEP 3: Loading Model...")

# Check Hardware
device_type = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Using Precision: {device_type}")

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=device_type)

# Stability Hack: Freeze model and force Norms/Output to Float32
for param in model.parameters():
    param.requires_grad = False  # Freeze everything first
    if param.ndim == 1:
        param.data = param.data.to(torch.float32) # Cast LayerNorm to FP32

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [ ]:
class CastOutputToFloat(nn.Sequential):
    def forward(self, x): return super().forward(x).to(torch.float32)
model.lm_head = CastOutputToFloat(model.lm_head)

In [ ]:
print(">>> STEP 4: Injecting LoRA Adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# This is the magic line that adds the "Green Badges" (Trainable Params)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
print(">>> STEP 5: Initializing Trainer...")

training_args = TrainingArguments(
    output_dir="./OPT-fine_tuned-LIMA",
    per_device_train_batch_size=4, # Reduced to 4 to be safe on T4 GPU
    gradient_accumulation_steps=2, # Accumulate to simulate batch size 8
    learning_rate=1e-4,
    num_train_epochs=1,            # Reduced to 1 for quick testing
    logging_steps=10,
    fp16=True if torch.cuda.is_available() else False, # Use FP16 on GPU
    save_strategy="no",            # Don't save checkpoints to save disk space
    report_to="none",              # Disable WandB for simplicity
    remove_unused_columns=False    # Important for custom datasets
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

In [ ]:
import torch
from peft import PeftModel

# ==========================================
# STEP 1: MERGE (With Safety Check)
# ==========================================
print(">>> Attempting to merge model...")

# We check if the model has the 'merge_and_unload' method
# This prevents the "AttributeError" if you run the cell twice
if hasattr(model, "merge_and_unload"):
    model = model.merge_and_unload()
    print("✅ Merge successful! LoRA adapters fused into base model.")
else:
    print("ℹ️ Model is already merged (or is standard). Skipping merge step.")

# ==========================================
# STEP 2: SAVE TO DISK
# ==========================================
save_folder = "./OPT-LIMA-Merged-Final"
print(f">>> Saving model to {save_folder}...")

# We set safe_serialization=False to fix the OPT "Shared Tensors" error
model.save_pretrained(save_folder, safe_serialization=False)
tokenizer.save_pretrained(save_folder)

print("✅ Save complete!")

# ==========================================
# STEP 3: FINAL TEST (INFERENCE)
# ==========================================
print(">>> Running Test Inference...")

# 1. Prepare Input
prompt = "Question: How do I make a cup of tea?\n\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt")

# 2. Move to GPU/CPU
device = model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

# 3. Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

# 4. Print Result
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("-" * 30)
print(generated_text)
print("-" * 30)